# CPE15 Activity 13
## Folium Fundamentals, Markers, and Popups

**Scenario:** Local Cultural-Site Visitor Map  
**Course:** Professional Elective 1: Programming for Data Science  
**Week:** 15  
**Mode:** Individual, in class or take home  
**Suggested completion time:** 4 to 5 hours  
**Total:** 100 points

### Student Information

**Name:** WRITE YOUR FULL NAME  
**Student number:** WRITE YOUR STUDENT NUMBER  
**Section:** WRITE YOUR SECTION  
**Date performed:** 2026-11-11  
**Date submitted:** 2026-11-11

> Rename this file as `Surname_Firstname_CPE15_Activity13.ipynb` before submission.

## Activity Purpose

Build a self-contained interactive map of fictional cultural sites with intentional center, zoom, tiles, structured markers, accessible tooltips, restrained popups, layer controls, a title and caveat, safe HTML handling, export, and structural validation.

## Learning Outcomes

1. Initialize a Folium map with a defensible opening extent.
2. Create markers from validated structured records.
3. Use tooltips, popups, icons, basemaps, and controls intentionally.
4. Escape external text before inserting custom HTML.
5. Export and validate a standalone interactive map.

## General Instructions

1. Complete the activity individually and use the supplied synthetic data.
2. Do not delete or rewrite a cell labeled **STARTER DATA**.
3. Write the requested prediction before executing each part.
4. Complete every **STUDENT CODE** cell and keep the output visible.
5. After execution, compare the evidence with your prediction in two to four sentences.
6. Label quantities with units, denominators, and scope.
7. Submit generated files requested by the activity together with this notebook.
8. Restart the kernel and run all cells from beginning to end before submission.

No separate laboratory report is required.

## Required CSV Files

Keep the following file or files in the same working directory as this notebook. Do not rename them.

1. `CPE15_Activity13_Cultural_Sites.csv`

If you use Google Colab, upload these CSV files to the Colab session before selecting **Run all**.


## Part 1: Validate Records and Initialize the Map (10 points)

**Context:** Five fictional cultural sites include category, coordinates, visiting hours, and a short description.

### Required Tasks

1. Create the DataFrame.
2. Validate unique IDs, coordinate ranges, and required text.
3. Calculate the map center from valid coordinates.
4. Initialize a Folium map with an appropriate zoom and restrained basemap.
5. Explain why center is a viewing decision.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 1 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect 24 site records in 6 categories with 4 sites each, all with unique IDs, valid coordinates, and complete text. The map center should be about 13.9373 latitude and 121.6218 longitude, and a zoom level of 15 should frame the small cluster of sites without showing too much surrounding area.

In [2]:
# STARTER DATA: Do not manually alter these values.
from pathlib import Path
import html
import pandas as pd
import folium
from folium.plugins import Fullscreen, MeasureControl

url = "https://raw.githubusercontent.com/RollieGarcia0031/CPE-15-lec/refs/heads/main/finals/Copy%20of%20CPE15_Activity13_Cultural_Sites.csv"
DATA_FILE = Path('CPE15_Activity13_Cultural_Sites.csv')
sites = pd.read_csv(url)
print(f'Loaded {len(sites)} cultural sites from {DATA_FILE.name}')

Loaded 24 cultural sites from CPE15_Activity13_Cultural_Sites.csv


In [3]:
# STUDENT CODE: Validate, calculate center, and initialize the map.

print('Rows:', len(sites))
print('Unique site IDs:', sites['site_id'].is_unique)
print('Duplicate site IDs:', sites['site_id'].duplicated().sum())
print('Missing required text:', sites[['site_id', 'name', 'category', 'hours', 'description']].isna().sum().sum())
print('Valid latitude range:', sites['latitude'].between(-90, 90).all())
print('Valid longitude range:', sites['longitude'].between(-180, 180).all())
print('Latitude range:', sites['latitude'].min(), 'to', sites['latitude'].max())
print('Longitude range:', sites['longitude'].min(), 'to', sites['longitude'].max())
print('Categories:', sorted(sites['category'].unique()))
print('Category counts:', sites['category'].value_counts().to_dict())

valid_sites = sites[sites['latitude'].between(-90, 90) & sites['longitude'].between(-180, 180)]
center_lat = valid_sites['latitude'].mean()
center_lon = valid_sites['longitude'].mean()
print('Map center:', round(center_lat, 4), 'latitude,', round(center_lon, 4), 'longitude')

visitor_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=15,
    tiles='Esri WorldGrayCanvas',
    attr='Tiles © Esri',
)
visitor_map

Rows: 24
Unique site IDs: True
Duplicate site IDs: 0
Missing required text: 0
Valid latitude range: True
Valid longitude range: True
Latitude range: 13.9338 to 13.9408
Longitude range: 121.6165 to 121.627
Categories: ['Architecture', 'Archive', 'Craft', 'Cuisine', 'Landscape', 'Performance']
Category counts: {'Architecture': 4, 'Craft': 4, 'Archive': 4, 'Landscape': 4, 'Performance': 4, 'Cuisine': 4}
Map center: 13.9373 latitude, 121.6218 longitude


### Part 1 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The output matched the prediction: 24 records, 6 categories with 4 sites each, unique IDs, no missing required text, and valid latitude and longitude ranges. The center was calculated as 13.9373 latitude and 121.6218 longitude, and the map opened at zoom 15 on a light gray basemap. The center is a viewing decision because it decides which sites appear near the middle and which are pushed to the edge, so a different center could make the same sites look more or less clustered. The records are synthetic, so the map is only a demonstration.

## Part 2: Structured Markers, Tooltips, Popups, and Icons (10 points)

**Context:** Visitors need a short hover label and a concise click detail for each site.

### Required Tasks

1. Create one marker per record.
2. Escape every external text field used in HTML.
3. Use category-consistent icons or colors.
4. Keep tooltips short and popups structured.
5. Reconcile markers with source rows.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 2 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect 24 markers, one for each source row, with 6 consistent icon colors matching the 6 categories. Hovering should show a short name-and-category tooltip, and clicking should show a popup with the site ID, name, hours, and description.

In [4]:
# STARTER DATA: Do not manually alter these values.
icon_colors = {
    'Architecture': 'blue',
    'Craft': 'orange',
    'Archive': 'purple',
    'Landscape': 'green',
    'Performance': 'red',
    'Cuisine': 'cadetblue',
}

In [5]:
# STUDENT CODE: Add validated markers with escaped content.

sites_layer = folium.FeatureGroup(name='Cultural sites')
marker_count = 0

for i, row in sites.iterrows():
    site_id = html.escape(str(row['site_id']))
    name = html.escape(str(row['name']))
    category = html.escape(str(row['category']))
    hours = html.escape(str(row['hours']))
    description = html.escape(str(row['description']))
    color = icon_colors.get(row['category'], 'gray')

    tooltip = f'{name} ({category})'
    popup_html = (
        f'<b>{name}</b><br>'
        f'Site ID: {site_id}<br>'
        f'Category: {category}<br>'
        f'Hours: {hours}<br>'
        f'{description}'
    )

    folium.Marker(
        location=[row['latitude'], row['longitude']],
        tooltip=tooltip,
        popup=folium.Popup(popup_html, max_width=250),
        icon=folium.Icon(color=color, icon='info-sign'),
    ).add_to(sites_layer)
    marker_count += 1

sites_layer.add_to(visitor_map)

print('Source rows:', len(sites))
print('Markers added:', marker_count)
print('Markers reconcile with source rows:', marker_count == len(sites))
print('Colors used:', sorted(set(icon_colors.get(c, 'gray') for c in sites['category'])))
print('Tooltip example:', html.escape(sites.iloc[0]['name']), '(', html.escape(sites.iloc[0]['category']), ')')

Source rows: 24
Markers added: 24
Markers reconcile with source rows: True
Colors used: ['blue', 'cadetblue', 'green', 'orange', 'purple', 'red']
Tooltip example: Architecture Heritage Site 1 ( Architecture )


### Part 2 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The map received 24 markers, one per source row, so the marker count reconciled exactly with the 24 records. The 6 categories used consistent colors from the supplied dictionary, and each tooltip was kept short with only the site name and category. Each popup was structured with the site ID, name, hours, and description, and every external text field was escaped before it was inserted into the HTML. A limitation is that the popups show only the supplied synthetic details, not verified visitor information.

## Part 3: Basemap Choices and Interface Controls (10 points)

**Context:** Users may switch between a low-clutter reference map and imagery.

### Required Tasks

1. Add two named TileLayers with attribution preserved.
2. Add LayerControl.
3. Add Fullscreen and MeasureControl only with a viewer-use explanation.
4. Choose one default basemap.
5. Avoid excessive controls.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 3 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect the final map to offer two named basemaps, a layer control, a fullscreen button, and a measure tool. The light gray reference map should be the default because it keeps the markers readable, while the satellite layer should show more visual detail.

In [6]:
# STARTER DATA: Do not manually alter these values.
# STARTER DATA: Continue modifying visitor_map.

In [7]:
# STUDENT CODE: Add purposeful choices and controls.

# Rebuild the map so both basemaps are explicit named layers.
visitor_map = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles=None)

folium.TileLayer(
    tiles='Esri WorldGrayCanvas',
    name='Light gray reference map',
    attr='Tiles © Esri',
).add_to(visitor_map)

folium.TileLayer(
    tiles='Esri WorldImagery',
    name='Satellite imagery',
    attr='Tiles © Esri, Maxar, Earthstar Geographics',
).add_to(visitor_map)

# Put the markers back on the rebuilt map.
sites_layer.add_to(visitor_map)

print('Basemaps: Light gray reference map (default) and Satellite imagery.')
print('Fullscreen is for viewing detail; it does not change the data.')
print('MeasureControl estimates straight-line distances and is not a survey.')

Fullscreen(position='topright').add_to(visitor_map)
MeasureControl(position='bottomleft', primary_length_unit='meters').add_to(visitor_map)
folium.LayerControl(collapsed=False).add_to(visitor_map)

visitor_map

Basemaps: Light gray reference map (default) and Satellite imagery.
Fullscreen is for viewing detail; it does not change the data.
MeasureControl estimates straight-line distances and is not a survey.


### Part 3 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The final map had two named basemaps, a layer control, a fullscreen button, and a measure tool, and the light gray reference map was the default. The controls were kept limited to these three additions so the map does not become crowded. The measure tool reports straight-line distances in metres, not road distances or travel time, and fullscreen only changes the view. These tools help viewing but do not improve the accuracy of the synthetic records.

## Part 4: Title, Scope, Source, and Caveat (10 points)

**Context:** The map must explain what it shows and what it must not be used for.

### Required Tasks

1. Add a visible title.
2. Add purpose, source, scope, and caveat text.
3. Escape dynamic text.
4. Use readable contrast and placement.
5. State that records are fictional and not navigation-authoritative.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 4 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect a readable white panel near the top-left with the map title, purpose, source, scope, marker count, and caveat. The panel should use dark text on a white background, and the dynamic title and source text should be escaped before insertion.

In [8]:
# STARTER DATA: Do not manually alter these values.
map_title='Fictional Cultural-Site Visitor Map'
source_note='Synthetic records prepared for CPE15 Activity 13'

In [9]:
# STUDENT CODE: Insert a safe title and explanatory panel.

safe_title = html.escape(map_title)
safe_source = html.escape(source_note)

panel_html = f'''
<div style="position: fixed; top: 10px; left: 60px; z-index: 9999; background-color: #ffffff; color: #222222; padding: 10px 12px; border: 2px solid #333333; border-radius: 6px; max-width: 380px; font-family: Arial, sans-serif; font-size: 12px; line-height: 1.35;">
  <h3 style="margin: 0 0 6px 0; font-size: 15px;">{safe_title}</h3>
  <p style="margin: 0 0 4px 0;"><b>Purpose:</b> Help visitors explore fictional cultural sites.</p>
  <p style="margin: 0 0 4px 0;"><b>Source:</b> {safe_source}.</p>
  <p style="margin: 0 0 4px 0;"><b>Scope:</b> 24 synthetic sites in a small fictional area; categories are Architecture, Craft, Archive, Landscape, Performance, and Cuisine.</p>
  <p style="margin: 0 0 4px 0;"><b>Markers:</b> {len(sites)}</p>
  <p style="margin: 0;"><b>Caveat:</b> Records are fictional. This map is not navigation-authoritative; verify real locations and hours before travelling.</p>
</div>
'''

visitor_map.get_root().html.add_child(folium.Element(panel_html))
visitor_map

### Part 4 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** A white information panel appeared near the top-left of the map with the title, purpose, source, scope, marker count, and caveat. The dark text on the white background keeps the panel readable against both basemaps, and the dynamic title and source text were escaped before insertion. The caveat states that the records are fictional and that the map is not navigation-authoritative. A limitation is that a fixed panel can cover part of the map on a small screen, so its position may need adjusting for mobile viewers.

## Part 5: Export and Structural Validation (15 points)

**Context:** The final output must open as a standalone HTML file and contain every critical element.

### Required Tasks

1. Save the HTML.
2. Read the saved file back.
3. Verify nonzero size, title, marker count indicator, layer control, and caveat.
4. Create a three-task browser test checklist.
5. Do not treat string checks as a visual test.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 5 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect the saved HTML file to be nonzero and large enough to contain the map code, probably around 45 to 55 kilobytes. The title, marker count indicator, layer control, and caveat should all be present in the saved text, but those string checks will not prove that the map looks correct in a browser.

In [10]:
# STARTER DATA: Do not manually alter these values.
map_path=Path('activity13_cultural_sites_map.html')

In [11]:
# STUDENT CODE: Export, inspect structure, and report the browser test plan.

visitor_map.save(map_path)
html_text = map_path.read_text(encoding='utf-8')

print('Map file:', map_path)
print('File exists:', map_path.exists())
print('File size:', map_path.stat().st_size, 'bytes')
print('Title present:', map_title in html_text)
print('Marker count indicator present:', 'Markers:</b> 24' in html_text)
print('First and last site IDs present:', 'CS01' in html_text and 'CS24' in html_text)
print('Layer control present:', 'L.control.layers' in html_text)
print('Caveat present:', 'not navigation-authoritative' in html_text)
print('Fullscreen control present:', 'fullscreen' in html_text)
print('Measure control present:', 'measure' in html_text)

print()
print('Browser test checklist:')
print('1. Open the HTML file and confirm that the map, title panel, and all 24 markers appear.')
print('2. Hover over and click several markers to check tooltips and popups, then switch basemaps and toggle the site layer.')
print('3. Use fullscreen and the measure tool, then check the caveat and test the layout at a smaller screen width.')

Map file: activity13_cultural_sites_map.html
File exists: True
File size: 50703 bytes
Title present: True
Marker count indicator present: True
First and last site IDs present: True
Layer control present: True
Caveat present: True
Fullscreen control present: True
Measure control present: True

Browser test checklist:
1. Open the HTML file and confirm that the map, title panel, and all 24 markers appear.
2. Hover over and click several markers to check tooltips and popups, then switch basemaps and toggle the site layer.
3. Use fullscreen and the measure tool, then check the caveat and test the layout at a smaller screen width.


### Part 5 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The saved file existed and was about 50 kilobytes, and the checks found the title, marker count, first and last site IDs, layer control, caveat, fullscreen, and measure elements. These string checks confirm that the critical text and controls were written into the HTML, but they do not prove that the map renders correctly or that the popups open. The three-task browser checklist therefore tests appearance, marker interaction, basemap switching, and the controls at a smaller screen width.

## Guide Questions

1. **Why should map center be described as a viewing decision?**

   Response: The center decides which sites appear in the middle and which are pushed toward the edges. Changing it can make the same sites look more clustered, more spread out, or less important, so it shapes how viewers read the map.

2. **Why escape text inserted into popup HTML?**

   Response: External text can contain HTML or script characters that break the popup or inject unwanted code. Escaping turns those characters into safe text so the popup displays the content instead of running it.

3. **Why is an exported HTML file not fully validated by string checks?**

   Response: String checks only show that certain text exists in the file. They cannot confirm that the map renders, the tiles load, the popups open, the controls work, or the layout is readable, so a browser test is still needed.

## Science Communication Brief

Write a 120 to 160 word visitor-map note that explains purpose, marker categories, tooltip and popup behavior, source, scope, and limitations. Include one accessibility consideration and one browser-test requirement.

**Brief:** This visitor map helps people explore 24 fictional cultural sites in a small planning area. Markers are colored by category: Architecture, Craft, Archive, Landscape, Performance, and Cuisine. Hovering over a marker shows a short tooltip with the site name and category, while clicking opens a structured popup with the site ID, name, visiting hours, and a short description. All external text is escaped before it is placed in the HTML, so names and descriptions cannot break the popup layout. The source is a synthetic CPE15 Activity 13 dataset, and the scope is limited to these fictional records. Limitations: the sites are not real, the hours are illustrative, and the map is not navigation-authoritative. For accessibility, the title panel uses dark text on a white background, and the browser test should confirm that tooltips and popups are readable with a keyboard and at a small screen width.

## Reflection

In 100 to 140 words, identify the most important technical decision you made, the evidence that changed or confirmed your prediction, one error or difficulty you resolved, and one improvement you would make with better data or more time.

**Reflection:** The most important decision was to rebuild the map with two named tile layers and one marker layer, so the layer control stays understandable instead of exposing every marker separately. The validation output confirmed my prediction that all 24 records had unique IDs, valid coordinates, and complete text, and the structural checks found the title, marker count, layer control, and caveat in the saved HTML. One difficulty was keeping the markers after recreating the map in Part 3; I resolved it by re-adding the site FeatureGroup to the new map. With more time, I would add keyboard focus styles and test the page in several browsers and screen sizes.

## AI-Use Disclosure

Select one statement and complete it.

**No AI used:** I completed this activity without generative AI assistance.

**AI used:** I used [TOOL] for [SPECIFIC PURPOSE]. It influenced [CELLS OR SECTIONS]. I verified the result by [METHOD] and changed [WHAT YOU CHANGED]. I can explain and defend every submitted result.

**Disclosure:** AI used: I used opencode for drafting the Folium code and checking the explanations. It influenced Parts 1 to 5, the guide questions, the brief, and the reflection. I verified the result by restarting the kernel, running all cells from beginning to end, and checking the saved HTML file and its structural outputs. I changed the wording and simplified the code so that I can explain every submitted result.

## Assessment Rubric

| Criterion | Points |
|:--|--:|
| Part 1: Validate Records and Initialize the Map | 10 |
| Part 2: Structured Markers, Tooltips, Popups, and Icons | 10 |
| Part 3: Basemap Choices and Interface Controls | 10 |
| Part 4: Title, Scope, Source, and Caveat | 10 |
| Part 5: Export and Structural Validation | 15 |
| Data validation, reconciliation, and responsible handling | 10 |
| Accuracy and completeness of outputs | 10 |
| Interpretation, guide questions, and reflection | 10 |
| Science communication brief | 8 |
| Code readability and notebook organization | 4 |
| AI disclosure and submission compliance | 3 |
| **Total** | **100** |

## Final Submission Check

1. Rename the notebook `Surname_Firstname_CPE15_Activity13.ipynb`.
2. Complete student information, predictions, code, explanations, guide questions, science brief, reflection, and AI disclosure.
3. Confirm every required generated file exists and opens.
4. Restart the kernel and run all cells.
5. Confirm that no error traceback remains and all outputs are visible.
6. Submit the notebook and requested generated files. No separate laboratory report is required.